# Sensor Placement Optimization — Colab + Isaac Sim

This notebook runs the **outer-loop optimizer** from this repo in Google Colab.

You can use these modes:

- **Mock mode (works everywhere)**: run with `--dummy` (alias for `mock_isaac` evaluator).
- **Isaac mode (this repo)**: set `inner_loop.mode: isaac_sim` and provide an `env` to `IsaacSimEvaluator` that implements:
  - `reconfigure_sensors(env_idx, config, sensor_models)`
  - `run_rollouts(n_episodes, rng) -> list[EvalMetrics]`

## Practical setup on Colab (two common options)

### A) **Same-VM “bridge” (recommended)**
- **Process 1 (Colab kernel / `python`)**: runs CMA-ES + `IsaacSimEvaluator`.
- **Process 2 (Isaac’s Python)**: runs Isaac + a tiny HTTP server on `http://127.0.0.1:<port>` that exposes the two methods above.

This is the most reliable way to connect Colab to Isaac, because `Isaac Sim` often ships and expects its own Python environment.

### B) **Remote Isaac**
If Isaac is not on the same machine as the notebook kernel, keep the same HTTP bridge, but run process 2 on the remote GPU box and make it reachable to Colab (public IP, tailscale, reverse SSH tunnel, etc.).

The cells below show (A) with a local HTTP server, plus a drop-in place to swap mock metrics for your real `SimulationApp` / scene logic.


In [ ]:
# --- 1) Get the code ---
# Option A (recommended): clone from GitHub
#   - Replace REPO_URL with your repo URL.
# Option B: if you uploaded the repo as a zip to Colab, unzip it and set REPO_DIR.

import os

REPO_URL = "https://github.com/dcaglar-28/sensor_placement_opt.git"  # change if you fork/clone a different origin
REPO_DIR = "/content/sensor_placement_opt"

if REPO_URL.startswith("http"):
    # NOTE: in Colab `!` shells, use `{REPO_DIR}` (Python), not `"$REPO_DIR"` (shell var).
    !rm -rf "{REPO_DIR}"
    !git clone "{REPO_URL}" "{REPO_DIR}"

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
print("contents:", os.listdir("."))


In [ ]:
# --- 2) Install dependencies (into THIS kernel's python) ---

import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])

print("python:", sys.executable)


In [ ]:
# --- 2b) Verify the Colab kernel can import the optimizer stack ---

import importlib
import sys

importlib.import_module("cma")
importlib.import_module("yaml")

import cma  # noqa: F401

print("OK: pip-installed deps are importable in this kernel")
print("python:", sys.executable)
print("cma:", cma.__version__)


In [ ]:
# --- 3) Quick sanity check: run in mock mode ---
# This should work on Colab without Isaac Sim.

!python -m sensor_opt.run_experiment --config configs/default.yaml --dummy --no-mlflow


## Colab with Isaac Sim on the same VM: HTTP bridge to `127.0.0.1`

If your Colab can launch Isaac, you typically still have **two Python environments**:
- the Colab `python` running this notebook, and
- the Isaac/Kit python used by Isaac Sim.

`IsaacSimEvaluator` is intentionally simple: it just needs an `env` object. The easiest way to connect those environments is a tiny local HTTP service.

The cell below:
- `pip install`s FastAPI/uvicorn
- starts a **mock** Isaac backend (`MockIsaacEnvManager`) as a *stand-in* for your real `SimulationApp` code
- runs one **tiny** CMA-ES pass to prove end-to-end wiring

**Replace the mock server body** with your real Isaac reconfiguration + rollout logic.


In [ ]:
# --- Local Isaac bridge (HTTP) + tiny smoke test ---

!python -m pip install -q -U "fastapi>=0.110,<1" "uvicorn[standard]>=0.24,<1"

import os
import sys
import threading
import time
from contextlib import contextmanager
from pathlib import Path

import uvicorn
import yaml
import copy
from fastapi import FastAPI
from pydantic import BaseModel
import numpy as np

from sensor_opt.inner_loop.isaac_env_manager import MockIsaacEnvManager
from sensor_opt.inner_loop.isaac_evaluator import IsaacSimEvaluator
from sensor_opt.logging.experiment_logger import ExperimentLogger
from sensor_opt.search.factory import create_search
from sensor_opt.encoding.config import SensorConfig, SingleSensorConfig


@contextmanager
def _background_uvicorn(app: FastAPI, host: str, port: int):
    config = uvicorn.Config(app, host=host, port=port, log_level="warning")
    server = uvicorn.Server(config)

    t = threading.Thread(target=server.run, daemon=True)
    t.start()

    # wait for bind
    deadline = time.time() + 30.0
    import socket

    while time.time() < deadline:
        try:
            with socket.create_connection((host, port), timeout=0.25):
                break
        except OSError:
            time.sleep(0.05)
    else:
        raise RuntimeError("Timed out waiting for uvicorn to start")

    try:
        yield
    finally:
        server.should_exit = True
        t.join(timeout=5.0)


class ReconfigureRequest(BaseModel):
    env_idx: int
    config: dict
    sensor_models: dict


class RunRolloutsRequest(BaseModel):
    n_episodes: int
    seed: int


def _sensor_config_from_payload(d: dict) -> SensorConfig:
    sensors = [SingleSensorConfig(**s) for s in d.get("sensors", [])]
    return SensorConfig(sensors=sensors)


# NOTE: swap MockIsaacEnvManager for your real Isaac-backed manager.
NUM_ENVS = 1
isaac = MockIsaacEnvManager(
    num_envs=NUM_ENVS,
    baseline_noise_std=0.01,
    stochastic_std=0.02,
)

app = FastAPI()


@app.post("/reconfigure_sensors")
def reconfigure_sensors(req: ReconfigureRequest):
    cfg = _sensor_config_from_payload(req.config)
    isaac.reconfigure_sensors(req.env_idx, cfg, req.sensor_models)
    return {"ok": True}


@app.post("/run_rollouts")
def run_rollouts(req: RunRolloutsRequest):
    rng = np.random.default_rng(int(req.seed))
    metrics = isaac.run_rollouts(n_episodes=int(req.n_episodes), rng=rng)
    return {
        "metrics": [
            {
                "collision_rate": float(m.collision_rate),
                "blind_spot_fraction": float(m.blind_spot_fraction),
                "mean_goal_success": float(m.mean_goal_success),
                "n_episodes": int(m.n_episodes),
            }
            for m in metrics
        ]
    }

HOST = "127.0.0.1"
PORT = 8010

# Import the client from earlier in the notebook; if you restarted runtime, re-run that cell.
from sensor_opt.loss.loss import EvalMetrics
from dataclasses import asdict
from typing import Any
import json
import urllib.request


class IsaacRpcEnvClient:
    """(Duplicated on purpose) Small JSON HTTP client for Colab runtimes that restart cells out-of-order."""

    def __init__(self, base_url: str, timeout_sec: float = 300.0):
        self.base_url = base_url.rstrip("/")
        self.timeout_sec = float(timeout_sec)

    def reconfigure_sensors(self, env_idx: int, config: SensorConfig, sensor_models: dict) -> None:
        payload = {"env_idx": int(env_idx), "config": asdict(config), "sensor_models": sensor_models}
        self._post_json("/reconfigure_sensors", payload)

    def run_rollouts(self, n_episodes: int, rng: np.random.Generator) -> list[EvalMetrics]:
        seed = int(rng.integers(0, np.iinfo(np.int32).max))
        payload = {"n_episodes": int(n_episodes), "seed": seed}
        data = self._post_json("/run_rollouts", payload)
        return [
            EvalMetrics(
                collision_rate=float(m["collision_rate"]),
                blind_spot_fraction=float(m["blind_spot_fraction"]),
                mean_goal_success=float(m["mean_goal_success"]),
                n_episodes=int(m["n_episodes"]),
            )
            for m in data["metrics"]
        ]

    def _post_json(self, path: str, payload: dict) -> Any:
        url = f"{self.base_url}{path}"
        req = urllib.request.Request(
            url,
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        with urllib.request.urlopen(req, timeout=self.timeout_sec) as resp:
            return json.loads(resp.read().decode("utf-8"))


with _background_uvicorn(app, host=HOST, port=PORT):
    cfg = yaml.safe_load(open("configs/default.yaml"))
    cfg = copy.deepcopy(cfg)
    cfg["inner_loop"]["mode"] = "isaac_sim"
    cfg["multi_fidelity"]["enabled"] = False  # keep the smoke test fast + single-fidelity

    cfg["cma"]["max_generations"] = 1
    cfg["cma"]["population_size"] = 4
    cfg["cma"]["tolx"] = 1e-9
    cfg["cma"]["tolfun"] = 1e-9

    env = IsaacRpcEnvClient(base_url=f"http://{HOST}:{PORT}", timeout_sec=600.0)
    base_evaluator = IsaacSimEvaluator(isaac_sim_cfg={"env": env, "num_envs": NUM_ENVS})
    seed = int(cfg.get("experiment", {}).get("seed", 42))

    with ExperimentLogger(
        experiment_name=cfg["experiment"]["name"] + "_isaac_smoke",
        results_dir=cfg.get("logging", {}).get("results_dir", "results"),
        use_mlflow=False,
        mlflow_tracking_uri=cfg.get("logging", {}).get("mlflow_tracking_uri", "mlruns"),
        run_config=cfg,
    ) as logger:
        search = create_search(
            search_type=cfg["search"]["type"],
            config=cfg,
            evaluator={
                "evaluator_fn": None,
                "evaluator": None,
                "base_evaluator": base_evaluator,
                "logger": logger,
                "seed": seed,
            },
        )
        result = search.run()

    print("smoke best_loss:", result.best_loss)
    print("smoke run_id:", result.run_id)


## Isaac Sim integration (what you need to implement)

Your repo already defines the expected Isaac-side API in:

- `sensor_opt/inner_loop/isaac_evaluator.py` (`IsaacSimEvaluator`)
- `sensor_opt/inner_loop/isaac_env_manager.py` (the `env` protocol)

### The contract
Your Isaac Sim process must provide an object (or RPC wrapper) with:

- `reconfigure_sensors(env_idx, config, sensor_models)`
- `run_rollouts(n_episodes, rng) -> list[EvalMetrics]`

Where `EvalMetrics` is:

- `collision_rate` (0..1)
- `blind_spot_fraction` (0..1)
- `mean_goal_success` (0..1)
- `n_episodes`

### Recommended architecture
Pick one:

- **Same VM (typical Colab + Isaac)**: run a small HTTP server next to Isaac (often easiest as a second process using Isaac’s Python), and call it from the notebook via `http://127.0.0.1:<port>`.
- **Remote GPU box**: same HTTP bridge, but expose it with a routable address (VPN / SSH tunnel / public endpoint).

Below is a **minimal HTTP client wrapper** you can adapt. The server can be FastAPI, Flask, gRPC-gateway, etc.


In [ ]:
# --- Isaac RPC adapter template (CLIENT side, runs in the notebook kernel) ---
# You will need a matching server process (often easiest: FastAPI running in Isaac's python, or a thread in the notebook for smoke tests).

from __future__ import annotations

from dataclasses import asdict
from typing import Any

import numpy as np

from sensor_opt.encoding.config import SensorConfig
from sensor_opt.loss.loss import EvalMetrics


class IsaacRpcEnvClient:
    """Client wrapper that matches the env API IsaacSimEvaluator expects."""

    def __init__(self, base_url: str, timeout_sec: float = 120.0):
        self.base_url = base_url.rstrip("/")
        self.timeout_sec = float(timeout_sec)

    def reconfigure_sensors(self, env_idx: int, config: SensorConfig, sensor_models: dict) -> None:
        payload = {
            "env_idx": int(env_idx),
            "config": asdict(config),
            "sensor_models": sensor_models,
        }
        self._post_json("/reconfigure_sensors", payload)

    def run_rollouts(self, n_episodes: int, rng: np.random.Generator) -> list[EvalMetrics]:
        # Send an explicit seed so the server can be deterministic if desired.
        seed = int(rng.integers(0, np.iinfo(np.int32).max))
        payload = {"n_episodes": int(n_episodes), "seed": seed}
        data = self._post_json("/run_rollouts", payload)

        # Expected server response:
        # {"metrics": [{"collision_rate":..., "blind_spot_fraction":..., "mean_goal_success":..., "n_episodes":...}, ...]}
        metrics = []
        for m in data["metrics"]:
            metrics.append(
                EvalMetrics(
                    collision_rate=float(m["collision_rate"]),
                    blind_spot_fraction=float(m["blind_spot_fraction"]),
                    mean_goal_success=float(m["mean_goal_success"]),
                    n_episodes=int(m["n_episodes"]),
                )
            )
        return metrics

    def _post_json(self, path: str, payload: dict) -> Any:
        # Keeping this dependency-light (requests is available in Colab, but we avoid pinning it).
        import json
        import urllib.request

        url = f"{self.base_url}{path}"
        req = urllib.request.Request(
            url,
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        with urllib.request.urlopen(req, timeout=self.timeout_sec) as resp:
            return json.loads(resp.read().decode("utf-8"))


In [ ]:
# --- Wiring the optimizer to IsaacSimEvaluator (example) ---
# Point this at whatever host is running the HTTP bridge:
# - local: http://127.0.0.1:<port>
# - remote: http://<gpu-box>:<port>

import copy
import yaml

from sensor_opt.inner_loop.isaac_evaluator import IsaacSimEvaluator
from sensor_opt.search.factory import create_search
from sensor_opt.logging.experiment_logger import ExperimentLogger


cfg = yaml.safe_load(open("configs/default.yaml"))
cfg = copy.deepcopy(cfg)

# Switch to Isaac mode
cfg["inner_loop"]["mode"] = "isaac_sim"
cfg["multi_fidelity"]["enabled"] = False  # optional: speeds up / simplifies evaluation plumbing

# Create the env client that IsaacSimEvaluator will call.
# If your HTTP bridge is running on the same VM, localhost works.
# If the bridge is remote, replace host/port accordingly (and ensure routing/firewall).
ISAAC_SERVER_URL = "http://127.0.0.1:8010"

NUM_ENVS = 1  # must match the bridge server's parallel env count

env = IsaacRpcEnvClient(base_url=ISAAC_SERVER_URL, timeout_sec=300.0)
base_evaluator = IsaacSimEvaluator(isaac_sim_cfg={"env": env, "num_envs": NUM_ENVS})

seed = int(cfg.get("experiment", {}).get("seed", 42))

with ExperimentLogger(
    experiment_name=cfg["experiment"]["name"],
    results_dir=cfg.get("logging", {}).get("results_dir", "results"),
    use_mlflow=False,
    mlflow_tracking_uri=cfg.get("logging", {}).get("mlflow_tracking_uri", "mlruns"),
    run_config=cfg,
) as logger:
    search = create_search(
        search_type=cfg["search"]["type"],
        config=cfg,
        evaluator={
            "evaluator_fn": None,
            "evaluator": None,
            "base_evaluator": base_evaluator,
            "logger": logger,
            "seed": seed,
        },
    )
    result = search.run()

print("best_loss:", result.best_loss)
print("run_id:", result.run_id)
